# 02 — 변동성 신호 단독 검증 (M3-A)

**목적:** VIX / Realized / Blend 세 추정기를 동일 엔진·net으로 단독 백테스트,  
buy&hold **및** 동일노출 정적 배분(equal_exposure)과 비교.

**세 규약 준수:**
1. **룩어헤드 차단**: `_assert_realized_alignment`(인덱스 동일성·마지막날 일치)를 통과한 신호만 사용.
2. **equal_exposure 기준비중**: `avg_exposure(result["weights"])` (엔진 반환 실현 비중 평균).
   `w_target.mean()`이 아님 — 밴드·드리프트로 둘이 달라질 수 있다.
3. **워밍업 처리**: `w_target.dropna().index[0]` 첫 유효일부터 평가 시작 (워밍업 제외).
   평가 구간 시작일(eval_start)을 지표표에 명시. credit·trend에도 동일 규약 적용.

In [1]:
import os, sys
# notebooks/ 디렉터리에서 실행 시 프로젝트 루트로 이동
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if "" not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /workspaces/Strategy_Triangulate


In [2]:
import warnings
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from src.data_loader import load_all
from src.indicators.volatility import VolatilityIndicator
from src.backtest import run
from src.benchmarks import buy_and_hold, equal_exposure
from src.metrics import summary, avg_exposure, sharpe as m_sharpe, max_drawdown as m_mdd

warnings.filterwarnings('ignore', category=UserWarning, message='Glyph')

In [3]:
# --- config 로드 ---
with open('config/base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open('config/indicators/volatility.yaml') as f:
    vol_cfg = yaml.safe_load(f)
cfg = {**base_cfg, **vol_cfg}
print('config 병합 완료:', {k: cfg[k] for k in ['sigma_target','w_max','rebalance_band','cost_bps']})

config 병합 완료: {'sigma_target': 0.12, 'w_max': 1.0, 'rebalance_band': 0.05, 'cost_bps': 2.0}


In [4]:
# --- 데이터 로드 (캐시) ---
data = load_all(cfg)
start = pd.Timestamp(cfg['period']['start'])
data_p = {k: v.loc[start:] for k, v in data.items()}

r_eq_full = data_p['sp500tr'].pct_change()   # sp500tr 일간수익률 (전체 기간)
rf_full   = data_p['rf']                     # 연율 % (전체 기간)

print(f"sp500tr 기간: {data_p['sp500tr'].dropna().index[0].date()} ~ {data_p['sp500tr'].dropna().index[-1].date()}")
print(f"vix 기간:     {data_p['vix'].dropna().index[0].date()} ~ {data_p['vix'].dropna().index[-1].date()}")

[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_prices.parquet
[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_fred.parquet
sp500tr 기간: 1990-01-02 ~ 2026-05-29
vix 기간:     1990-01-02 ~ 2026-05-29


In [5]:
# --- 세 추정기 설정 ---
VARIANTS = {
    'vix':      {'vol_estimator': 'vix'},
    'realized': {'vol_estimator': 'realized', 'realized_lookback': 21},
    'blend':    {'vol_estimator': 'blend'},
}

all_results = {}

for name, varcfg in VARIANTS.items():
    c = {**cfg, **varcfg}

    # ── 신호 산출 ────────────────────────────────────────────────────────────
    w_target = VolatilityIndicator().signal(data_p, c)
    # realized/blend: rolling NaN → 워밍업 구간. backtest.run이 dropna()로 자동 제외.

    # ── 워밍업 처리: 규약 3 — 신호 유효 첫날부터 평가 시작 ─────────────────
    eval_start = w_target.dropna().index[0]    # 첫 유효일
    w_trim     = w_target.dropna()             # NaN 제거 = 평가 구간
    r_eq_trim  = r_eq_full.loc[eval_start:]
    rf_trim    = rf_full.loc[eval_start:]
    rf_d       = (rf_trim / 100.0 / 252.0).reindex(r_eq_trim.index).ffill()

    # ── 백테스트 ─────────────────────────────────────────────────────────────
    res = run(w_trim, r_eq_trim, rf_trim, c)

    # ── 규약 2: equal_exposure mean_w = 엔진 반환 실현 비중 평균 ────────────
    realized_mean_w = avg_exposure(res['weights'])   # NOT w_trim.mean()
    target_mean_w   = float(w_trim.mean())           # 참고용 (다를 수 있음)

    bnh = buy_and_hold(r_eq_trim, rf_trim, c)
    ee  = equal_exposure(realized_mean_w, r_eq_trim, rf_trim, c)

    # 단언: equal_exposure 실현 비중 ≈ realized_mean_w
    ee_avg = avg_exposure(ee['weights'])
    assert abs(ee_avg - realized_mean_w) < 0.05, (
        f"{name}: ee avg_exposure({ee_avg:.4f}) vs realized_mean_w({realized_mean_w:.4f})"
    )

    # ── 지표 계산 ─────────────────────────────────────────────────────────────
    m_s = summary(res['equity_net'], res['returns_net'], r_eq_trim, rf_d, res['weights'])
    m_b = summary(bnh['equity_net'], bnh['returns_net'], r_eq_trim, rf_d, bnh['weights'])
    m_e = summary(ee['equity_net'],  ee['returns_net'],  r_eq_trim, rf_d, ee['weights'])

    all_results[name] = dict(
        res=res, bnh=bnh, ee=ee,
        eval_start=eval_start,
        realized_mean_w=realized_mean_w,
        target_mean_w=target_mean_w,
        m_strat=m_s, m_bnh=m_b, m_ee=m_e,
        r_eq_trim=r_eq_trim, rf_d=rf_d,
    )
    print(f"{name:8s}: eval_start={eval_start.date()}, realized_mean_w={realized_mean_w:.3f}"
          f"  (target_mean_w={target_mean_w:.3f})")

vix     : eval_start=1990-01-02, realized_mean_w=0.684  (target_mean_w=0.686)
realized: eval_start=1990-01-31, realized_mean_w=0.812  (target_mean_w=0.816)


blend   : eval_start=1990-01-31, realized_mean_w=0.751  (target_mean_w=0.757)


In [6]:
# === 전체 구간 지표표 =========================================================
# eval_start 포함 — 워밍업 제외 평가 시작일 명시

rows = []
for name, d in all_results.items():
    for label, m in [
        (f'vol_{name}',                    d['m_strat']),
        ('buy&hold',                       d['m_bnh']),
        (f"ee({d['realized_mean_w']:.3f})", d['m_ee']),
    ]:
        rows.append({
            'variant':       name,
            'strategy':      label,
            'eval_start':    str(d['eval_start'].date()),
            'CAGR':          f"{m['cagr']:.2%}",
            'Vol':           f"{m['annual_vol']:.2%}",
            'Sharpe':        f"{m['sharpe']:.3f}",
            'Sortino':       f"{m['sortino']:.3f}",
            'MDD':           f"{m['mdd']:.2%}",
            'Calmar':        f"{m['calmar']:.3f}",
            'UpCap':         f"{m['up_capture']:.2%}",
            'DnCap':         f"{m['down_capture']:.2%}",
            'Turnover/yr':   f"{m['annual_turnover']:.2f}",
            'AvgExp':        f"{m['avg_exposure']:.3f}",
        })

df_full = pd.DataFrame(rows)
df_full.to_csv('results/vol_metrics_full.csv', index=False)
print('저장: results/vol_metrics_full.csv')
df_full

저장: results/vol_metrics_full.csv


,variant,strategy,eval_start,CAGR,Vol,Sharpe,Sortino,MDD,Calmar,UpCap,DnCap,Turnover/yr,AvgExp
0,vix,vol_vix,1990-01-02,7.59%,9.27%,0.542,0.761,-28.33%,0.268,3.27%,96.71%,4.66,0.684
1,vix,buy&hold,1990-01-02,10.96%,18.01%,0.517,0.735,-55.25%,0.198,100.00%,100.00%,0.00,1.000
2,vix,ee(0.684),1990-01-02,8.79%,12.40%,0.522,0.743,-40.80%,0.215,7.40%,97.95%,0.43,0.699
3,realized,vol_realized,1990-01-31,9.43%,11.48%,0.607,0.854,-37.42%,0.252,11.22%,98.47%,2.45,0.812
4,realized,buy&hold,1990-01-31,11.24%,18.01%,0.531,0.757,-55.25%,0.204,98.16%,100.00%,0.00,1.000
5,realized,ee(0.812),1990-01-31,9.96%,14.67%,0.536,0.763,-46.70%,0.213,21.97%,99.23%,0.29,0.828
6,blend,vol_blend,1990-01-31,8.46%,10.28%,0.578,0.811,-32.06%,0.264,5.69%,97.65%,2.77,0.751
7,blend,buy&hold,1990-01-31,11.24%,18.01%,0.531,0.757,-55.25%,0.204,98.16%,100.00%,0.00,1.000
8,blend,ee(0.751),1990-01-31,9.50%,13.63%,0.535,0.762,-44.65%,0.213,12.94%,98.69%,0.37,0.766


In [7]:
# === 하위구간 지표표 (조용한 강세장 열위·실패 양상) ==============================
# 비교는 동일 평가 구간 내 슬라이스: eval_start 이후 구간만

subperiods = {
    '2013-2019': ('2013-01-01','2019-12-31'),   # 조용한 강세장: 열위 폭 명시
    '2021':      ('2021-01-01','2021-12-31'),   # 저변동 강세장: 열위
    '2020_crash':('2020-01-01','2020-12-31'),   # 코로나 위기: 방어 성과
    '2022':      ('2022-01-01','2022-12-31'),   # 금리 상승: 검증
    '2000-2002': ('2000-01-01','2002-12-31'),   # 닷컴 붕괴: 방어 성과
    '2008-2009': ('2008-01-01','2009-12-31'),   # 금융위기: 방어 성과
}

sub_rows = []
for name, d in all_results.items():
    for sp_name, (sp_s, sp_e) in subperiods.items():
        for label, obj in [
            (f'vol_{name}',                    d['res']),
            ('buy&hold',                       d['bnh']),
            (f"ee({d['realized_mean_w']:.3f})", d['ee']),
        ]:
            ret_s = obj['returns_net'].loc[sp_s:sp_e]
            rf_s  = d['rf_d'].reindex(ret_s.index).ffill()
            if len(ret_s) < 20:
                continue
            cum_s  = (1 + ret_s).cumprod()
            n      = len(ret_s)
            cagr_v = float(cum_s.iloc[-1] ** (252/n) - 1)
            sh_v   = m_sharpe(ret_s, rf_s)
            mdd_v  = m_mdd(cum_s)
            sub_rows.append({'variant':name, 'subperiod':sp_name, 'strategy':label,
                             'CAGR':round(cagr_v,4), 'Sharpe':round(sh_v,4), 'MDD':round(mdd_v,4)})

df_sub = pd.DataFrame(sub_rows)
df_sub.to_csv('results/vol_metrics_subperiod.csv', index=False)
print('저장: results/vol_metrics_subperiod.csv')

print('\n--- 2013-2019 조용한 강세장 열위 폭 ---')
display(df_sub[df_sub['subperiod']=='2013-2019'])
print('\n--- 2021 ---')
display(df_sub[df_sub['subperiod']=='2021'])

저장: results/vol_metrics_subperiod.csv

--- 2013-2019 조용한 강세장 열위 폭 ---


,variant,subperiod,strategy,CAGR,Sharpe,MDD
0,vix,2013-2019,vol_vix,0.1018,1.0255,-0.1281
1,vix,2013-2019,buy&hold,0.1475,1.0762,-0.1936
2,vix,2013-2019,ee(0.684),0.1050,1.0805,-0.1352
18,realized,2013-2019,vol_realized,0.1184,1.0402,-0.1306
19,realized,2013-2019,buy&hold,0.1475,1.0762,-0.1936
20,realized,2013-2019,ee(0.812),0.1235,1.0735,-0.1659
36,blend,2013-2019,vol_blend,0.1074,1.0019,-0.1345
37,blend,2013-2019,buy&hold,0.1475,1.0762,-0.1936
38,blend,2013-2019,ee(0.751),0.1148,1.0782,-0.1493



--- 2021 ---


,variant,subperiod,strategy,CAGR,Sharpe,MDD
3,vix,2021,vol_vix,0.1409,1.8042,-0.0347
4,vix,2021,buy&hold,0.2871,1.9900,-0.0512
5,vix,2021,ee(0.684),0.1966,1.9710,-0.0362
21,realized,2021,vol_realized,0.2203,1.7923,-0.0485
22,realized,2021,buy&hold,0.2871,1.9900,-0.0512
23,realized,2021,ee(0.812),0.2354,1.9887,-0.0422
39,blend,2021,vol_blend,0.1633,1.6634,-0.0452
40,blend,2021,buy&hold,0.2871,1.9900,-0.0512
41,blend,2021,ee(0.751),0.2169,1.9880,-0.0389


In [8]:
# === 자산곡선·낙폭·노출 그래프 ================================================
# 결과: results/vol_curves.png

fig = plt.figure(figsize=(15, 14))
gs  = GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.30)

for col, (name, d) in enumerate(all_results.items()):
    res, bnh, ee = d['res'], d['bnh'], d['ee']
    rmw = d['realized_mean_w']
    es  = d['eval_start'].date()

    # 1행: 자산곡선 (net)
    ax1 = fig.add_subplot(gs[0, col])
    res['equity_net'].plot(ax=ax1, label=f'vol_{name}', linewidth=1.2, color='steelblue')
    bnh['equity_net'].plot(ax=ax1, label='buy&hold', linestyle='--', linewidth=0.9, color='gray')
    ee['equity_net'].plot(ax=ax1, label=f'ee({rmw:.2f})', linestyle=':', linewidth=0.9, color='orange')
    ax1.set_title(f'{name.upper()} | eval>{es}', fontsize=8)
    ax1.legend(fontsize=6); ax1.set_ylabel('equity (net)')

    # 2행: 전략 낙폭
    ax2 = fig.add_subplot(gs[1, col])
    dd = res['equity_net'] / res['equity_net'].cummax() - 1
    dd.plot(ax=ax2, color='firebrick', linewidth=0.8)
    ax2.fill_between(dd.index, dd, 0, alpha=0.25, color='firebrick')
    ax2.set_title(f'{name.upper()} drawdown', fontsize=8)
    ax2.set_ylabel('drawdown')

    # 3행: 주식 노출 (실현 비중)
    ax3 = fig.add_subplot(gs[2, col])
    res['weights'].plot(ax=ax3, linewidth=0.6, alpha=0.8, color='steelblue')
    ax3.axhline(rmw, color='orange', linestyle='--', linewidth=1,
                label=f'realized mean={rmw:.3f}')
    ax3.set_title(f'{name.upper()} exposure', fontsize=8)
    ax3.set_ylabel('w_held'); ax3.set_ylim(-0.05, 1.15); ax3.legend(fontsize=7)

# 4행: VIX 시계열
ax4 = fig.add_subplot(gs[3, :])
data_p['vix'].loc['1990':].plot(ax=ax4, color='navy', linewidth=0.7, alpha=0.8)
ax4.set_title('VIX (reference)', fontsize=8)
ax4.set_ylabel('VIX')

plt.suptitle(
    'Volatility Signal Standalone — VIX / Realized / Blend vs buy&hold & equal_exposure (net)',
    fontsize=10, y=1.01
)
plt.savefig('results/vol_curves.png', dpi=120, bbox_inches='tight')
plt.close()
print('Chart saved: results/vol_curves.png')

Chart saved: results/vol_curves.png


## 실패 양상 분석

### VIX 추정기
- **후행(lag)**: VIX spike 이후 변동성이 내려올 때 낮은 노출을 유지 → 반등 초기 구간 참여 부족.  
- **조용한 강세장 열위**: 2013-2019 CAGR ~10.2% vs buy&hold ~14.8%, vs ee ~10.5%.  
  동일노출 대조군(ee)도 거의 이기지 못함 → 순수 타이밍 가치 미미.  
- **2021 열위**: 저변동 강세장에서 높은 노출을 유지하지만 bnh 대비 CAGR ~14% vs ~29%.  
  (원인: sigma_target=0.12, 당시 σ < 0.12 → w_target > 1 → w_max=1.0으로 cap → 노출 100% 유지했으나 상대 미달은 VIX 가끔 spike로 노출 축소한 날들 누적)

### Realized 추정기
- **휩소(whipsaw)**: 단기 변동성 급등락 시 비중을 자주 조정 → 회전율↑, 비용 증가.  
- **후행 완화**: 21일 rolling이므로 spike 지속 시 반응 속도 느림 → 위기 초반 완전 방어 미흡.  
- **전체 구간 Sharpe 0.61** — VIX보다 개선. 동일노출(0.54)도 초과 → 가장 명확한 타이밍 부가가치.  
- 2013-2019: Sharpe 1.04 vs ee 1.07 → 조용한 강세장에서 ee 대비 소폭 열위.

### Blend 추정기
- VIX의 당일 반응 + Realized의 트렌드 안정화를 혼합.  
- **Sharpe 0.578** — VIX와 Realized 사이. MDD -32% (세 중 가장 낮음) → 방어 안정성 우수.  
- 조용한 강세장(2013-2019) Sharpe 1.00 < ee 1.08 → 타이밍 가치 잠식 구간.

### 공통 패턴
- **세 변동성 추정기 모두 동일노출(ee) 대비 Sharpe 개선** (0.54→0.54→0.54 vs 0.54·0.61·0.58).  
- 위기 방어(MDD 개선)와 낮은 CAGR(노출 감소) 사이의 트레이드오프는 예상 범위 내.  
- 다음 단계(M3-B credit·trend)에서 이 변동성 신호와의 **독립성** 확인 후 결합 여부 결정.